In [1]:
import sys

import itertools

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from IPython.display import clear_output

In [2]:
sys.path.append("../model-collection/")
from mttrans import RL_gru, get_optimizer, get_scheduler

sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
model_name = "mttrans"
utr_type = "utr5"
device_id = 0 if utr_type == "utr5" else 1
seqsize = 50 if utr_type == "utr5" else 240

In [4]:
PATH_FROM = f"../../predictor/regression_multiple/{utr_type.upper()}_zscores_replicateagg.csv"
df = pd.read_csv(PATH_FROM)

In [5]:
n_splits = 10  # Remaining splits for the 'train' set

df['fold'] = df['fold'].map({'train': -1, 'val': n_splits - 2, 'test': n_splits - 1})

df_unsplit = df.loc[df['fold'] == -1, ['seq', 'fold']].copy()
df_unsplit['fold'] = df_unsplit['seq'].map(
    {seq: split_no
     for seq, split_no in
     zip(df_unsplit['seq'].drop_duplicates(),
         itertools.cycle(range(n_splits - 2)))
     }
)
df.loc[df_unsplit.index, 'fold'] = df_unsplit['fold']

In [6]:
fold_sizes = df['fold'].value_counts()
fold_size = fold_sizes.max()
fold_sizes

fold
0    10805
1    10805
2    10805
3    10805
4    10805
5    10805
6    10805
7    10805
8    10805
9    10800
Name: count, dtype: int64

In [7]:
num_classes = df["cell_type"].unique().shape[0]
num_classes

5

In [8]:
splits = dict(tuple(df.groupby('fold')))
for split_df in splits.values():
    split_df.reset_index(drop=True, inplace=True)
splits[8].head()

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
0,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c1,8,68.449816,60.870873,47.590087,41.838243,2.287164,2.29667,-0.009506,-0.062351,0.152462
1,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c17,8,72.812510,53.335530,34.751567,42.078913,2.227102,2.29667,-0.069567,-0.456294,0.152462
2,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c2,8,71.853249,67.896625,57.267869,55.146572,2.379545,2.29667,0.082875,0.543581,0.152462
3,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c4,8,77.312032,66.148593,64.039833,77.299136,2.496230,2.29667,0.199560,1.308917,0.152462
4,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c6,8,86.538512,56.970286,42.241073,32.322653,2.093308,2.29667,-0.203362,-1.333853,0.152462


In [9]:
batch_size = 32
train_size = fold_size * (n_splits - 2)
steps_per_epoch = max(1, int(train_size // batch_size))

epochs = None

In [10]:
num_workers = 32

In [11]:
folds_arg = [{
    'train_folds': {j % n_splits for j in range(i, i + n_splits - 2)},
    'val_fold': (i - 2) % n_splits,
    'test_fold': (i - 1) % n_splits
} for i in range(n_splits)]
folds_arg

[{'train_folds': {0, 1, 2, 3, 4, 5, 6, 7}, 'val_fold': 8, 'test_fold': 9},
 {'train_folds': {1, 2, 3, 4, 5, 6, 7, 8}, 'val_fold': 9, 'test_fold': 0},
 {'train_folds': {2, 3, 4, 5, 6, 7, 8, 9}, 'val_fold': 0, 'test_fold': 1},
 {'train_folds': {0, 3, 4, 5, 6, 7, 8, 9}, 'val_fold': 1, 'test_fold': 2},
 {'train_folds': {0, 1, 4, 5, 6, 7, 8, 9}, 'val_fold': 2, 'test_fold': 3},
 {'train_folds': {0, 1, 2, 5, 6, 7, 8, 9}, 'val_fold': 3, 'test_fold': 4},
 {'train_folds': {0, 1, 2, 3, 6, 7, 8, 9}, 'val_fold': 4, 'test_fold': 5},
 {'train_folds': {0, 1, 2, 3, 4, 7, 8, 9}, 'val_fold': 5, 'test_fold': 6},
 {'train_folds': {0, 1, 2, 3, 4, 5, 8, 9}, 'val_fold': 6, 'test_fold': 7},
 {'train_folds': {0, 1, 2, 3, 4, 5, 6, 9}, 'val_fold': 7, 'test_fold': 8}]

In [12]:
checked = {
    "seed": [3],
    "folds": folds_arg,
    "features": [
        ("sequence", "conditions"),
    ],
    "augment_dict": [
        dict(
            extend_left=0,
            extend_right=0,
            shift_left=0,
            shift_right=0,
            revcomp=False,
        ),
    ],
    "epochs": [100],
}

In [13]:
def launch_model(
    seed: int,
    train_ds_kws: dict,
    val_ds_kws: dict,
    model_class,
    model_kws: dict,
    criterion_class,
    criterion_kws: dict,
    optimizer_class,
    optimizer_kws: dict,
    lr_scheduler_class,
    lr_scheduler_kws: dict,
    test_time_validation: bool,
    train_folds: set,
    val_fold: int,
    test_fold: int,
    initialize_weights: bool,
    epochs: int = epochs,
):
    pl.seed_everything(seed)

    # Creating Datasets
    train_set = utrdata.UTRData(
        df=pd.concat([splits[i] for i in train_folds]),
        **train_ds_kws,
    )
    val_set = utrdata.UTRData(
        df=splits[val_fold],
        **val_ds_kws,
    )
    test_set = utrdata.UTRData(
        df=splits[test_fold],
        **val_ds_kws,
    )

    assert train_set.num_channels == val_set.num_channels
    try:
        div_factor = val_ds_kws["augment_kws"]["shift_left"] + \
                     val_ds_kws["augment_kws"]["shift_right"] + 1
    except KeyError:
        div_factor = 1

    # Creating DataLoaders
    dl_train = DataLoader(
        train_set,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True
    )
    dl_val = DataLoader(
        val_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )
    dl_test = DataLoader(
        test_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )

    model = RNARegressor(
        model_class=model_class,
        model_kws=model_kws | dict(
            in_channels=train_set.num_channels
        ),
        criterion_class=criterion_class,
        criterion_kws=criterion_kws,
        optimizer_class=optimizer_class,
        optimizer_kws=optimizer_kws,
        lr_scheduler_class=lr_scheduler_class,
        lr_scheduler_kws=lr_scheduler_kws,
        test_time_validation=test_time_validation,
        initialize_weights=initialize_weights,
    )
    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        dirpath="saved_models",
        save_top_k=1,
        save_last=False,
        monitor="val_pearson_r_0",
        mode="max"
    )
    progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)

    logger = pl.loggers.tensorboard.TensorBoardLogger("tb_logs", name=model.model_name)
    trainer = pl.Trainer(
        callbacks=[checkpoint_callback, progressbar_callback],
        logger=logger,
        accelerator="gpu",
        devices=[device_id],
        # deterministic=True,
        max_epochs=epochs,
        num_sanity_val_steps=0,
        # gradient_clip_val=1e-5,
        # gradient_clip_algorithm="norm",
    )
    trainer.fit(model=model, train_dataloaders=dl_train, val_dataloaders=dl_val)
    best_model = RNARegressor.load_from_checkpoint(checkpoint_callback.best_model_path)

    prediction = trainer.predict(model=best_model, dataloaders=dl_test)
    test_pred, test_real = zip(*prediction)
    test_pred = torch.concat(test_pred).numpy()
    test_real = torch.concat(test_real).numpy()
    test_df = splits[test_fold].copy()
    test_df["real_0"] = test_real[:, 0]
    test_df["pred_0"] = test_pred[:, 0]

    return trainer, test_df

In [ ]:
results = []
for subset in itertools.product(
    *checked.values()
):
    PARAMS = dict(zip(checked.keys(), subset))
    AUGMENT_KEY = any(PARAMS["augment_dict"].values())
    AUGMENT_TEST_TIME = AUGMENT_KEY

    trainer_last, prediction_best_last = launch_model(
        seed=PARAMS["seed"],
        train_ds_kws=dict(
            construct_type=utr_type,
            predict_cols=["mass_center"],
            features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
            augment=AUGMENT_KEY,
            augment_test_time=False,
            augment_kws=PARAMS["augment_dict"],
        ),
        val_ds_kws=dict(
            construct_type=utr_type,
            predict_cols=["mass_center"],
            features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
            augment=False,
            augment_test_time=AUGMENT_TEST_TIME,
            augment_kws=PARAMS["augment_dict"],
        ),
        model_class=RL_gru,
        model_kws=dict(
            in_channels=5,
            conv_args=(
                [128, 256, 256, 256],
                3,
                [1, 2, 2, 2],
                [1, 1, 1, 1],
                [1, 1, 1, 1],
                0
            ),
            tower_width=80,
            dropout_rate=0.2,
            activation='ReLU',
        ),
        criterion_class=nn.MSELoss,
        criterion_kws=dict(),
        **get_optimizer(lr=0.003, l2=1e-06),
        **get_scheduler(),
        test_time_validation=AUGMENT_TEST_TIME,
        **PARAMS["folds"],
        initialize_weights=False,
        epochs=PARAMS["epochs"],
    )
    results.append((subset, prediction_best_last))

Global seed set to 3
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:612: UserWarning: Checkpoint directory /home/arsen_l/rna-legnet/wp6_clean_repo/parade-private/benchmark/crossval/saved_models exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type    | Params
--------------------------------------
0 | mod

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

## Validation

In [ ]:
full_result = pd.concat([r for _, r in results])
full_result

In [ ]:
full_result.to_csv(f"results/{utr_type}_{model_name}_crossval.csv", index=False)

In [ ]:
full_result.groupby('cell_type')[['mass_center', 'pred_0']].corr().unstack(-1)[('mass_center', 'pred_0')]

In [ ]:
full_result.groupby('seq')[['mass_center', 'pred_0']].mean().corr().stack()[('mass_center', 'pred_0')]

In [ ]:
full_result.groupby(['cell_type', 'fold'])[['mass_center', 'pred_0']].corr().unstack(-1)[('mass_center', 'pred_0')].unstack(0)

In [ ]:
full_result.groupby(['seq', 'fold'])[['mass_center', 'pred_0']].mean().groupby(level='fold').corr().unstack(-1)[('mass_center', 'pred_0')]

---